# racp

> Ramabana as an [Agent Client Protocol](https://agentclientprotocol.com/) agent: an editor
> drives this host, with its own files and its own terminal.

In [ ]:
#| default_exp racp

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import base64, typing
from fastcore.test import test_eq
from ramabana.testing import MemHost

In [ ]:
#| export
import asyncio, base64, os, sys, uuid
try:
    import acp
    from acp.helpers import update_available_commands
    from acp.schema import (AgentCapabilities, AvailableCommand, Implementation, InitializeResponse,
                            LoadSessionResponse, NewSessionResponse, PermissionOption, PromptCapabilities,
                            PromptResponse, ToolCallLocation, ToolCallUpdate)
# `ramabana-acp` installs whether or not the extra did, and an editor launching it shows the agent
# failing to start with whatever reached stderr. Name the extra there.
except ImportError as e: raise ImportError(
    f"ramabana-acp needs the acp extra: pip install 'ramabana[acp]' ({e})") from None
from fastcore.basics import ifnone
from fastcore.script import call_parse
from ramabana import __version__
from ramabana.agent import DFLT_TIMEOUT, Agent, Approvals
from ramabana.core import accepts
from ramabana.tools import WRITE_TOOLS, LocalHost

## What maps onto what

The protocol's shape is already this harness's shape, which is why this is a seam and not a
second harness. `session/prompt` is a turn. The `session/update` chunks are `Agent.stream`.
Tool calls are `Activity`. `session/request_permission` is `Approvals`, so the editor becomes
the person the gate waits for -- and because Ramabana refuses a write when nothing is
listening, losing the editor fails closed rather than open. `session/load` is
`resume_session`, so a conversation started in the terminal continues in the editor.

Two things run the other way. A client that offers `fs/read_text_file` knows about buffers
that are not on disk yet, and one that offers `terminal/*` can run a command where the person
can watch it. Reading and running through the editor is the difference between working on
what they see and working on a stale copy.

In [ ]:
#| export
#: `Act.kind` and a bare tool name, both onto the ten kinds ACP knows
KIND = {'search': 'search', 'view': 'read', 'edit': 'edit', 'web': 'fetch', 'run': 'execute',
        'delegate': 'think', 'memory': 'other', 'skill': 'other', 'tool': 'other'}
TOOL = {'edit_file': 'edit', 'replace_text': 'edit', 'create_file': 'edit', 'edit_cell': 'edit',
        'add_cell': 'edit', 'run_python': 'execute', 'run_shell': 'execute'}
PLAN = {'pending': 'pending', 'active': 'in_progress', 'done': 'completed', 'cancelled': 'completed'}
SHELL = os.environ.get('SHELL') or '/bin/sh'

OPTIONS = [PermissionOption(option_id='allow_once', name='Allow', kind='allow_once'),
           PermissionOption(option_id='allow_always', name='Allow every write this session',
                            kind='allow_always'),
           PermissionOption(option_id='reject_once', name='Reject', kind='reject_once')]

## The bridge

Every host method and every tool runs on the turn's worker thread; the protocol lives on an
event loop. One object owns the crossing, so nothing else has to think about it.

In [ ]:
#| export
class Bridge:
    "The one crossing between the turn's thread and the client's event loop."

    def __init__(self,
                 conn,                      # the ACP `Client` this agent is connected to
                 sid,                       # the session every call is about
                 loop,                      # the loop the connection is served on
                 timeout=DFLT_TIMEOUT):     # seconds to wait on a person, or on the editor
        self.conn, self.sid, self.loop, self.timeout = conn, sid, loop, timeout
        self.on_terminal = None    # called with a terminal id, so a tool call can show it

    def call(self, coro, timeout=None):
        "Block this thread on one round trip, and raise whatever the client raised."
        fut = asyncio.run_coroutine_threadsafe(coro, self.loop)
        try: return fut.result(ifnone(timeout, self.timeout))
        except Exception:
            # or the request sits in the connection's pending table for the rest of the session
            fut.cancel()
            raise

    def post(self, coro):
        "An update nothing waits on."
        return asyncio.run_coroutine_threadsafe(coro, self.loop)

## The host the editor answers for

`EditorHost` is `LocalHost` with four methods pointed at the client when the client said it
can serve them. Reading falls back to disk rather than failing, because an editor that
declines a read is not a reason to lose the turn. Writing does not: a write the editor
*refused* is an answer, and routing around it is the thing delegating the write prevents.
Nor does a command once its terminal has opened, because by then it has already run.

`read` is the load-bearing one. `view_file` reads through `Host.read`, so the view a model
addresses its edits against and the text those edits are applied to are one text. Without
that they would be two, and every hash would miss.

In [ ]:
#| export
class EditorHost(LocalHost):
    "`LocalHost` whose file text and shell come from the editor, where the editor offers them."

    def __init__(self, *args, **kw):
        super().__init__(*args, **kw)
        self.editor, self.can_read, self.can_write, self.can_run = None, False, False, False

    def attach(self, editor, fs=None, terminal=False):
        "Point this host at a `Bridge`, and at what the client said it can do."
        self.editor = editor
        self.can_read = bool(getattr(fs, 'read_text_file', False))
        self.can_write = bool(getattr(fs, 'write_text_file', False))
        self.can_run = bool(terminal)
        return self

    def _ok(self, what): return self.editor is not None and getattr(self, f'can_{what}')

    def read(self, path):
        if not self._ok('read'): return super().read(path)
        e = self.editor
        try: p = self.check(path, reading=True)
        except Exception: return None
        # an unsaved new file is not on disk, so a failed read here is the editor's answer,
        # not a reason to stop: fall back rather than lose the turn
        try: return e.call(e.conn.read_text_file(session_id=e.sid, path=str(p))).content
        except Exception: return super().read(path)

    def text_at(self, path):
        # a notebook is read as its cell sources, which only the base host knows how to do
        if str(path).endswith('.ipynb') or not self._ok('read'): return super().text_at(path)
        got = self.read(path)
        return super().text_at(path) if got is None else got

    def write(self, path, text):
        # no fallback here, unlike `read`. A read that the editor cannot serve costs nothing to
        # answer from disk; a write it *refused* must be reported, not routed around behind it
        if not self._ok('write'): return super().write(path, text)
        e, p = self.editor, self.check(path)
        e.call(e.conn.write_text_file(session_id=e.sid, path=str(p), content=str(text)))
        return str(p)

    def run_cmd(self, command, cwd=None, timeout=120):
        cmd = str(command or '').strip()
        # `tools_for` asks "can you run commands?" with an empty one, and must not spawn anything
        if not cmd: return 0, ''
        if not self._ok('run'): return super().run_cmd(command, cwd=cwd, timeout=timeout)
        opened = []
        try: return self._in_editor(cmd, cwd, timeout, opened)
        except Exception as e:
            # falling back once the terminal exists would run the command a second time
            if opened: return 1, f'the editor terminal failed after starting the command ({e!r})'
            return super().run_cmd(command, cwd=cwd, timeout=timeout)

    def _in_editor(self, cmd, cwd, timeout, opened):
        "Run it in the editor's terminal, so the person watches it live and keeps the scrollback."
        e, c = self.editor, self.editor.conn
        where = str(self.check(cwd)) if cwd else str(self.roots[0])
        # a shell, not the bare string: `run_cmd` promises pipes and redirects work
        t = e.call(c.create_terminal(session_id=e.sid, command=SHELL, args=['-c', cmd], cwd=where))
        tid = t.terminal_id
        opened.append(tid)
        try:
            if e.on_terminal: e.on_terminal(tid)
            done = e.call(c.wait_for_terminal_exit(session_id=e.sid, terminal_id=tid), timeout + 30)
            out = e.call(c.terminal_output(session_id=e.sid, terminal_id=tid))
            body = out.output + ('\n[output truncated by the editor]' if out.truncated else '')
            return ifnone(done.exit_code, 1), body
        finally:
            try: e.call(c.release_terminal(session_id=e.sid, terminal_id=tid))
            except Exception: pass

In [ ]:
#| export
def mk_agent(roots, model=None, approve='ask', web=True, vault=False, timeout=DFLT_TIMEOUT, **kw):
    "An `EditorHost` over `roots` and a gated `Agent` on it, without the terminal frontend's imports."
    approvals = Approvals(tools=WRITE_TOOLS, mode=approve, timeout=timeout)
    bases = [EditorHost]
    if vault:
        from ramabana.vault import VaultHost
        bases.append(VaultHost)
    Host = bases[0] if len(bases) == 1 else type('EditorVaultHost', tuple(bases), {})
    # read_outside stays off: an editor never names a path outside the folders it opened
    host = Host(list(roots), approvals=approvals, web=web, read_outside=False)
    approvals.host = host
    a = Agent(host, model=model, approvals=approvals, project_extensions=False, **kw)
    a.lend_model()
    return a, host

## The prompt

ACP content blocks arrive as a list. Text is joined; an image or a sound becomes the bytes a
model content part is made of. Pictures reach any model. Sound only reaches one rishi says can
hear it, and audio that cannot be sent is dropped with a note rather than in silence.

In [ ]:
#| export
def _res(b):
    "An embedded resource as text, whichever half of the union it is."
    r = getattr(b, 'resource', None)
    if (t := getattr(r, 'text', None)) is not None:
        return f'<file uri="{getattr(r, "uri", "")}">\n{t}\n</file>'
    return f'[binary resource {getattr(r, "uri", "")} ({getattr(r, "mimeType", "?")})]'

def blocks(prompt, spec=None):
    "ACP content blocks as one message: `(text, media)`, media being what `spec` can be sent."
    text, media = [], []
    for b in prompt:
        kind = getattr(b, 'type', '')
        if kind == 'text': text.append(b.text)
        elif kind in ('image', 'audio'):
            # `accepts` says yes where rishi cannot say, so this only drops what it knows
            # cannot be sent. Dropping with a reason beats the turn dying inside the engine
            if spec is None or accepts(spec, kind): media.append(base64.b64decode(b.data))
            else: text.append(f'[{kind} dropped: {b.mime_type} -- this model does not accept {kind}]')
        elif kind == 'resource': text.append(_res(b))
        elif kind == 'resource_link': text.append(f'@{getattr(b, "uri", "")}')
    return '\n\n'.join(t for t in text if t), media

## The session

One session is one agent. Everything the editor sees comes from hooks the harness already
has: `on_activity` for the tool calls, `on_plan` for the checklist, `Approvals.listen` for the
gate.

In [ ]:
#| export
class Session:
    "One ACP session: an agent, and the editor it reports to."

    def __init__(self, roots, conn, loop, sid=None, model=None, mk=None, timeout=DFLT_TIMEOUT, **kw):
        self.conn, self.br = conn, None
        self.seen, self.cancelled, self.gated, self.shell = set(), False, {}, ''
        self.agent, self.host = ifnone(mk, mk_agent)(roots, model=model, approve='ask',
                                                     timeout=timeout, on_activity=self._act, **kw)
        # the harness's own session id, so `session/load` can name a conversation and mean it.
        # A uuid could never match one, and `resume_session` would silently take the newest
        self.sid = sid or self.agent.session_id
        self.br = Bridge(conn, self.sid, loop, timeout)
        self.br.on_terminal = self._terminal
        # nothing listening means Ramabana refuses a write, so this registration is what
        # makes writes possible at all -- and losing it fails closed
        self.unhook = self.agent.approvals.listen(self._ask)
        self.agent.on_plan = self._plan

    def attach(self, caps):
        "Point the host at the editor, given what the client said in `initialize`."
        if isinstance(self.host, EditorHost):
            self.host.attach(self.br, getattr(caps, 'fs', None), getattr(caps, 'terminal', False))
        return self

    def _send(self, update):
        return None if self.br is None else self.br.post(self.conn.session_update(self.sid, update))

    @staticmethod
    def _key(tool, args): return (tool, str((args or {}).get('path', '')))

    def _terminal(self, tid):
        "The editor's terminal, shown inside the tool call that started it."
        if self.shell: self._send(acp.update_tool_call(self.shell, content=[acp.tool_terminal_ref(tid)]))

    def _act(self, a):
        d = a.dict()
        k = self._key(d['tool'], d['args'])
        # a gated call already has an entry, opened when permission was asked. Reusing its id
        # keeps the dialog and the tool call one thing in the editor rather than two
        tid = self.gated.get(k, d['id'])
        if d['tool'] == 'run_shell': self.shell = '' if d['done'] else tid
        where = [ToolCallLocation(path=p)] if (p := d['args'].get('path')) else None
        if tid not in self.seen:
            self.seen.add(tid)
            self._send(acp.start_tool_call(tid, d['summary'] or d['tool'],
                                           kind=KIND.get(d['kind'], 'other'), status='in_progress',
                                           locations=where, raw_input=d['args']))
        else: self._send(acp.update_tool_call(tid, status='in_progress'))
        if d['done']:
            self.gated.pop(k, None)
            body = [acp.tool_content(acp.text_block(d['detail']))] if d['detail'] else None
            self._send(acp.update_tool_call(tid, status='completed' if d['ok'] else 'failed',
                                            content=body))
        elif tid == d['id']:
            # not gated, so `_permit` never sent the diff. `a.args` rather than the clipped copy
            if (diff := self._body(d['tool'], a.args, '')) is not None:
                self._send(acp.update_tool_call(tid, content=diff))

    def _plan(self, plan):
        self._send(acp.update_plan([acp.plan_entry(t.text, status=PLAN.get(t.status, 'pending'))
                                    for t in plan.todos]))

    def _body(self, tool, args, preview):
        "A write the editor can render as a diff where the whole new text is known; else the preview."
        path = (args or {}).get('path', '')
        if tool == 'create_file' and path: return [acp.tool_diff_content(path, args.get('text', ''))]
        return [acp.tool_content(acp.text_block(preview))] if preview else None

    def _ask(self, a):
        "On the turn's thread, inside `Approvals.request`, before it waits."
        try: ok, note, always = self.br.call(self._permit(a))
        except Exception as e: ok, note, always = False, f'the editor did not answer ({e!r})', False
        answered = self.agent.approvals.answer(a.id, ok, note, session=always)
        # a refused call never runs, and a cancel may have answered it while the dialog was
        # open. Either way `_act` will not fire, so the entry has to be closed and dropped here
        if not ok or answered is None:
            self._send(acp.update_tool_call(a.id, status='failed'))
            self.gated.pop(self._key(a.tool, a.args), None)

    async def _permit(self, a):
        path = (a.args or {}).get('path', '')
        self.gated[self._key(a.tool, a.args)] = a.id
        self.seen.add(a.id)
        body = self._body(a.tool, a.args, a.preview)
        await self.conn.session_update(self.sid, acp.start_tool_call(
            a.id, a.summary or a.tool, kind=TOOL.get(a.tool, 'other'), status='pending',
            content=body, locations=[ToolCallLocation(path=path)] if path else None, raw_input=a.args))
        tc = ToolCallUpdate(tool_call_id=a.id, title=a.summary or a.tool,
                            kind=TOOL.get(a.tool, 'other'), raw_input=a.args, content=body)
        r = await self.conn.request_permission(session_id=self.sid, tool_call=tc, options=OPTIONS)
        # the option id, not the discriminator: a client serialising with `exclude_defaults`
        # drops `outcome`, and only an allowed outcome ever carries an option id
        oid = getattr(r.outcome, 'option_id', '') or ''
        if not oid: return False, 'the editor cancelled the request', False
        return oid.startswith('allow'), f'{oid} in the editor', oid == 'allow_always'

    def run(self, text, media=()):
        "The turn, on a worker thread. Chunks go back over the loop as they arrive."
        got = []
        for c in (self.agent.stream_with(text, image=list(media)) if media else self.agent.stream(text)):
            got.append(c)
            self._send(acp.update_agent_message_text(c))
        return ''.join(got)

    def commands(self):
        # the SDK's own helper: it sets the discriminator, without which the union will not serialise
        c = self.agent.commands
        return update_available_commands(AvailableCommand(name=n, description=f'/{n}')
                                         for n in (c() if callable(c) else c))

    def close(self):
        self.unhook()
        try: self.agent.close()
        except Exception: pass

## The agent

`initialize` is where the two directions are settled: what this agent can be sent, and what
the client can be asked for. The capabilities the client declares there are what
`EditorHost.attach` acts on, so a client that offers nothing gets a plain local host and
nothing breaks.

In [ ]:
#| export
class AcpAgent(acp.Agent):
    "Ramabana behind the Agent Client Protocol. One agent per session, rooted where the editor says."

    def __init__(self, model=None, roots=('.',), mk=None, timeout=DFLT_TIMEOUT, **kw):
        self.model, self.roots, self.mk, self.kw = model, list(roots), mk, kw
        self.timeout = timeout
        self.sessions, self.conn, self.caps = {}, None, None

    def on_connect(self, conn): self.conn = conn

    def _sess(self, sid):
        if (s := self.sessions.get(sid)) is None: raise acp.RequestError.invalid_params(f'no session {sid}')
        return s

    async def initialize(self, protocol_version, client_capabilities=None, client_info=None, **kw):
        self.caps = client_capabilities
        return InitializeResponse(
            protocol_version=min(protocol_version, acp.PROTOCOL_VERSION),
            agent_capabilities=AgentCapabilities(
                load_session=True,
                prompt_capabilities=PromptCapabilities(image=True, audio=True, embedded_context=True)),
            agent_info=Implementation(name='ramabana', title='Ramabana', version=__version__))

    async def _open(self, cwd, extra=None, sid=None):
        roots = [cwd, *(extra or [])] if cwd else list(self.roots)
        s = Session(roots, self.conn, asyncio.get_running_loop(), sid, self.model, self.mk,
                    self.timeout, **self.kw)
        self.sessions[s.sid] = s.attach(self.caps)
        await self.conn.session_update(s.sid, s.commands())
        return s

    async def new_session(self, cwd, additional_directories=None, mcp_servers=None, **kw):
        # `mcp_servers` is not honoured: this agent brings its own tools rather than the
        # editor's. An editor that configured some gets them silently ignored, not an error
        return NewSessionResponse(session_id=(await self._open(cwd, additional_directories)).sid)

    async def load_session(self, cwd, session_id, mcp_servers=None, additional_directories=None, **kw):
        "Resume the named conversation and replay it, so what the terminal started the editor continues."
        if (s := self.sessions.get(session_id)) is None:
            s = await self._open(cwd, additional_directories, session_id)
            try: picked = s.agent.resume_session(session_id)
            except Exception as e:
                # never fall back to 'latest': that hands the editor whichever conversation
                # happened to run last, from whichever project, and changes the model with it
                self.sessions.pop(s.sid, None)
                s.close()
                raise acp.RequestError.resource_not_found(f'no saved session {session_id} ({e})')
            got = picked['id']
        else: got = s.agent.session_id
        # only this conversation: `agent.history` is the whole log, every session in it
        for turn in [t for t in s.agent.history if t.get('session') == got]:
            if turn.get('prompt'):
                await self.conn.session_update(s.sid, acp.update_user_message_text(str(turn['prompt'])))
            if turn.get('reply'):
                await self.conn.session_update(s.sid, acp.update_agent_message_text(str(turn['reply'])))
        return LoadSessionResponse()

    async def prompt(self, session_id, prompt, **kw):
        s = self._sess(session_id)
        s.cancelled, s.seen = False, set()
        text, media = blocks(prompt, s.agent.model)
        if not text and not media: return PromptResponse(stop_reason='end_turn')
        s.gated = {}
        if text.startswith('/') and not media:
            # off the loop: /compact calls a model, and a command may reach the approval gate
            out = await asyncio.to_thread(s.agent.command, text) or f'unknown command {text.split()[0]}'
            await self.conn.session_update(session_id, acp.update_agent_message_text(out))
            return PromptResponse(stop_reason='end_turn')
        await asyncio.to_thread(s.run, text, media)
        return PromptResponse(stop_reason='cancelled' if s.cancelled else 'end_turn')

    async def cancel(self, session_id, **kw):
        if (s := self.sessions.get(session_id)) is None: return
        s.cancelled = True
        s.agent.cancel()

    async def close_session(self, session_id, **kw):
        if (s := self.sessions.pop(session_id, None)) is not None: s.close()

In [ ]:
#| export
async def serve(agent=None):
    "Speak ACP on stdio until the editor closes it."
    # stdout is the protocol. The streams take the real one first; after that anything that
    # prints -- a model loader, a warning -- goes to stderr, where it cannot corrupt a frame
    reader, writer = await acp.stdio_streams()
    sys.stdout = sys.stderr
    a = ifnone(agent, AcpAgent())
    try: await acp.run_agent(a, input_stream=writer, output_stream=reader)
    finally:
        # the editor closing does not stop a turn: it is on a worker thread, and the process
        # would sit there running tools until it finished. Cancel, then release the backends
        for s in list(a.sessions.values()):
            try: s.agent.cancel()
            except Exception: pass
            s.close()
        a.sessions.clear()

@call_parse
def main(
    root: str = '.',        # folders to open when the editor names none, comma separated
    model: str = None,      # the turn model. Omit for the routing default
    web: bool = True,       # let the web tools reach the network through fossick
    vault: bool = False,    # keep what is read in a vishalakshi vault, for the next session
    cfg: str = None,        # config dir, for skills, extensions and history
):
    "Serve Ramabana over the Agent Client Protocol, the way an editor launches an agent."
    from pathlib import Path
    roots = [r.strip() for r in str(root).split(',') if r.strip()]
    asyncio.run(serve(AcpAgent(model=model, roots=roots, web=web, vault=vault,
                               cfg=Path(cfg).expanduser() if cfg else None)))

## Tests

The wire itself is tested in `tests/test_acp.py`, which spawns this agent as a subprocess and
speaks JSON-RPC to it. What is worth reading here is the mapping, and the host.

In [ ]:
text, media = blocks([acp.text_block('why does'), acp.text_block('it fail?')])
test_eq(text, 'why does\n\nit fail?')
test_eq(media, [])

In [ ]:
#: an image reaches the model as the bytes a content part is made of
png = b'\x89PNG\r\n\x1a\n'
text, media = blocks([acp.text_block('what is this'), acp.image_block(base64.b64encode(png).decode(), 'image/png')])
test_eq(media, [png])
test_eq(text, 'what is this')

In [ ]:
#: every kind this harness names has somewhere to go in the editor
import typing
from acp.schema import ToolCallStart
known = set(typing.get_args(typing.get_args(ToolCallStart.model_fields['kind'].annotation)[0]))
test_eq(set(KIND.values()) - known, set())
test_eq(set(TOOL.values()) - known, set())
test_eq(set(PLAN.values()), {'pending', 'in_progress', 'completed'})

An `EditorHost` with nothing attached is a `LocalHost`, and the probe `tools_for` uses to
ask whether commands can be run must never reach the editor:

In [ ]:
h = EditorHost(['.'])
test_eq((h.can_read, h.can_write, h.can_run), (False, False, False))
test_eq(h.run_cmd(''), (0, ''))